## 02g_process_ipps_final_rule — RCM Intelligence Platform
### Purpose
Transforms raw Bronze IPPS Final Rule data into a clean, enriched
Silver table with year-over-year payment impacts calculated and ready
for strategic financial planning in Gold.

### What this does
1. Reads raw IPPS Final Rule data from Bronze
2. Casts all columns to correct data types (Bronze stores as string)
3. Calculates year-over-year payment changes (FY2024 vs FY2023)
4. Engineers financial impact metrics (wage index, DSH, GME, uncompensated care)
5. Enriches with hospital dimension data
6. Runs data quality checks with quarantine
7. Writes to Silver Delta table

### Outputs
- rcm_platform.rcm_silver.ipps_payment_impacts

### Notes
- Silver = Bronze + types + calculations + enrichment
- Failed DQ rows go to quarantine table — never silently dropped
- Annual refresh pattern (overwrite each fiscal year)

| Field      | Value |
|------------|-------|
| Author     | Mayank Joshi |
| Layer      | Silver |
| Run order  | After 01g_ingest_ipps_final_rule |
| Depends on | 00_config, 00_utils, Bronze IPPS table |

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_config

In [0]:
%run /Users/jmayank574@gmail.com/rcm-intelligence-platform/00_setup/00_utils

In [0]:
# ============================================================
# BATCH METADATA
# ============================================================

import uuid
from datetime import datetime, timezone

BATCH_ID        = str(uuid.uuid4())
BATCH_DATE      = datetime.now(timezone.utc).strftime("%Y-%m-%d")
BATCH_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
NOTEBOOK_NAME   = "02g_process_ipps_final_rule"

print(f"Batch ID        : {BATCH_ID}")
print(f"Batch date      : {BATCH_DATE}")
print(f"Batch timestamp : {BATCH_TIMESTAMP}")

In [0]:
# ============================================================
# STEP 1 — READ BRONZE AND CAST TO CORRECT TYPES
# Bronze stores everything as string — Silver applies types
# ============================================================

from pyspark.sql.functions import expr, col, trim, when, round as spark_round

print("=" * 55)
print("  READING BRONZE IPPS FINAL RULE")
print("=" * 55)

df_bronze = spark.table(f"{BRONZE_DB}.ipps_final_rule_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows     : {bronze_count:,}")
print(f"Bronze columns  : {len(df_bronze.columns)}")
print(f"Fiscal year     : {df_bronze.select('_fiscal_year').first()[0]}")

In [0]:
# ============================================================
# STEP 2 — BUILD TYPED IPPS DATAFRAME
# Cast all columns to appropriate types
# Using try_cast to handle empty strings gracefully
# ============================================================

print("\n" + "=" * 55)
print("  CASTING TO CORRECT DATA TYPES")
print("=" * 55)

df_typed = df_bronze.select(
    # Provider identifiers
    trim(col("provider_number")).alias("provider_number"),
    trim(col("name")).alias("hospital_name"),
    
    # Geographic identifiers
    trim(col("geographic_labor_market_area")).alias("geographic_cbsa"),
    trim(col("pre_reclass_labor_market_area")).alias("pre_reclass_cbsa"),
    trim(col("post_reclass_labor_market_area")).alias("post_reclass_cbsa"),
    trim(col("payment_labor_market_area")).alias("payment_cbsa"),
    trim(col("fips_county_code")).alias("fips_county_code"),
    expr("try_cast(region as int)").alias("region"),
    
    # Urban/Rural classification
    trim(col("urgeo")).alias("urban_rural_geo"),
    trim(col("urspa")).alias("urban_rural_spa"),
    trim(col("reclass")).alias("wage_index_reclassified"),
    
    # Wage index
    expr("try_cast(fy_2024_wage_index as decimal(10,6))").alias("fy_2024_wage_index"),
    trim(col("lugar")).alias("lugar_flag"),
    trim(col("section_401_hospital")).alias("section_401_hospital"),
    trim(col("section_505_eligible")).alias("section_505_eligible"),
    expr("try_cast(section_505_adjustment as decimal(10,6))").alias("section_505_adjustment"),
    expr("try_cast(cost_of_living_adjustment as decimal(10,6))").alias("cost_of_living_adjustment"),
    
    # Hospital characteristics
    expr("try_cast(resident_to_bed_ratio as decimal(10,6))").alias("resident_to_bed_ratio"),
    expr("try_cast(rday as int)").alias("resident_days"),
    expr("try_cast(beds as int)").alias("beds"),
    expr("try_cast(average_daily_census as int)").alias("average_daily_census"),
    expr("try_cast(bills as int)").alias("bills"),
    
    # Operating and Capital payments (FY2023 baseline = "40")
    expr("try_cast(tchop as decimal(18,2))").alias("fy2023_total_operating_payment"),
    expr("try_cast(tchcp as decimal(18,2))").alias("fy2023_total_capital_payment"),
    expr("try_cast(caseta40 as decimal(18,2))").alias("fy2023_operating_payment"),
    expr("try_cast(cmiv40 as decimal(18,4))").alias("fy2023_operating_case_mix_index"),
    expr("try_cast(tacmiv40 as decimal(18,2))").alias("fy2023_operating_case_adjusted_payment"),
    expr("try_cast(ime_caseta40 as decimal(18,2))").alias("fy2023_operating_ime_payment"),
    expr("try_cast(ime_tacmiv40 as decimal(18,2))").alias("fy2023_operating_ime_case_adjusted_payment"),
    
    # Operating and Capital payments (FY2024 final rule = "41")
    expr("try_cast(caseta41 as decimal(18,2))").alias("fy2024_operating_payment"),
    expr("try_cast(cmiv41 as decimal(18,4))").alias("fy2024_operating_case_mix_index"),
    expr("try_cast(tacmiv41 as decimal(18,2))").alias("fy2024_operating_case_adjusted_payment"),
    expr("try_cast(ime_caseta41 as decimal(18,2))").alias("fy2024_operating_ime_payment"),
    expr("try_cast(ime_tacmiv41 as decimal(18,2))").alias("fy2024_operating_ime_case_adjusted_payment"),
    
    # DSH (Disproportionate Share Hospital)
    expr("try_cast(dshpct as decimal(10,6))").alias("dsh_percentage"),
    expr("try_cast(dshopp as decimal(18,2))").alias("fy2024_dsh_operating_payment"),
    expr("try_cast(dshcpp as decimal(18,2))").alias("fy2024_dsh_capital_payment"),
    expr("try_cast(dsh_ly as decimal(18,2))").alias("fy2023_dsh_payment"),
    
    # Uncompensated Care Pool
    expr("try_cast(ucp_adj as decimal(18,2))").alias("fy2024_ucp_adjustment"),
    expr("try_cast(ucp_per_claim_amount as decimal(18,4))").alias("fy2024_ucp_per_claim_amount"),
    expr("try_cast(ucp_adj_ly as decimal(18,2))").alias("fy2023_ucp_adjustment"),
    expr("try_cast(ucp_per_claim_amount_ly as decimal(18,4))").alias("fy2023_ucp_per_claim_amount"),
    
    # Cost-to-charge ratios
    expr("try_cast(operating_ccr as decimal(10,6))").alias("operating_ccr"),
    expr("try_cast(capital_ccr as decimal(10,6))").alias("capital_ccr"),
    
    # Provider characteristics
    expr("try_cast(provider_type as int)").alias("provider_type_code"),
    expr("try_cast(hsp_rate as decimal(18,2))").alias("hsp_rate"),
    expr("try_cast(gaf as decimal(10,6))").alias("geographic_adjustment_factor"),
    expr("try_cast(capital_cost_of_living_adjustment as int)").alias("capital_cola_flag"),
    
    # Outlier factors
    expr("try_cast(outfact_f as decimal(10,6))").alias("operating_outlier_factor"),
    expr("try_cast(coutfact_f as decimal(10,6))").alias("capital_outlier_factor"),
    
    # Payer mix
    expr("try_cast(medicare_percentage as decimal(10,6))").alias("medicare_percentage"),
    expr("try_cast(medicaid_percentage as decimal(10,6))").alias("medicaid_percentage"),
    
    # Audit metadata
    col("_source"),
    col("_batch_id"),
    col("_batch_date"),
    col("_ingested_at"),
    col("_fiscal_year")
)

print(f"Rows after casting : {df_typed.count():,}")
print(f"Columns            : {len(df_typed.columns)}")

In [0]:
# ============================================================
# STEP 3 — CALCULATE YEAR-OVER-YEAR PAYMENT IMPACTS
# FY2024 vs FY2023 changes (the core value of this dataset)
# ============================================================

print("\n" + "=" * 55)
print("  CALCULATING PAYMENT IMPACTS")
print("=" * 55)

df_impacts = df_typed \
    .withColumn("operating_payment_change_dollars",
        when((col("fy2024_operating_payment").isNotNull()) & (col("fy2023_operating_payment").isNotNull()),
            spark_round(col("fy2024_operating_payment") - col("fy2023_operating_payment"), 2)
        ).otherwise(None)
    ) \
    .withColumn("operating_payment_change_pct",
        when((col("fy2023_operating_payment").isNotNull()) & (col("fy2023_operating_payment") > 0),
            spark_round(
                ((col("fy2024_operating_payment") - col("fy2023_operating_payment")) / col("fy2023_operating_payment")) * 100,
                2
            )
        ).otherwise(None)
    ) \
    .withColumn("total_payment_fy2024",
        when((col("fy2024_operating_payment").isNotNull()),
            spark_round(
                col("fy2024_operating_payment") + 
                when(col("fy2024_dsh_operating_payment").isNotNull(), col("fy2024_dsh_operating_payment")).otherwise(0) +
                when(col("fy2024_ucp_adjustment").isNotNull(), col("fy2024_ucp_adjustment")).otherwise(0),
                2
            )
        ).otherwise(None)
    ) \
    .withColumn("total_payment_fy2023",
        when((col("fy2023_operating_payment").isNotNull()),
            spark_round(
                col("fy2023_operating_payment") + 
                when(col("fy2023_dsh_payment").isNotNull(), col("fy2023_dsh_payment")).otherwise(0) +
                when(col("fy2023_ucp_adjustment").isNotNull(), col("fy2023_ucp_adjustment")).otherwise(0),
                2
            )
        ).otherwise(None)
    ) \
    .withColumn("total_payment_change_dollars",
        when((col("total_payment_fy2024").isNotNull()) & (col("total_payment_fy2023").isNotNull()),
            spark_round(col("total_payment_fy2024") - col("total_payment_fy2023"), 2)
        ).otherwise(None)
    ) \
    .withColumn("total_payment_change_pct",
        when((col("total_payment_fy2023").isNotNull()) & (col("total_payment_fy2023") > 0),
            spark_round(
                ((col("total_payment_fy2024") - col("total_payment_fy2023")) / col("total_payment_fy2023")) * 100,
                2
            )
        ).otherwise(None)
    ) \
    .withColumn("dsh_payment_change_dollars",
        when((col("fy2024_dsh_operating_payment").isNotNull()) & (col("fy2023_dsh_payment").isNotNull()),
            spark_round(col("fy2024_dsh_operating_payment") - col("fy2023_dsh_payment"), 2)
        ).otherwise(None)
    ) \
    .withColumn("ucp_payment_change_dollars",
        when((col("fy2024_ucp_adjustment").isNotNull()) & (col("fy2023_ucp_adjustment").isNotNull()),
            spark_round(col("fy2024_ucp_adjustment") - col("fy2023_ucp_adjustment"), 2)
        ).otherwise(None)
    ) \
    .withColumn("wage_index_impact_category",
        when(col("operating_payment_change_pct") > 2, "Positive Impact (>2%)")
        .when(col("operating_payment_change_pct").between(-2, 2), "Minimal Impact (-2% to 2%)")
        .when(col("operating_payment_change_pct") < -2, "Negative Impact (<-2%)")
        .otherwise("Unknown")
    ) \
    .withColumn("is_safety_net_hospital",
        when((col("dsh_percentage") > 0.25) | (col("medicaid_percentage") > 0.35), 1).otherwise(0)
    ) \
    .withColumn("is_teaching_hospital",
        when((col("resident_to_bed_ratio") > 0.1) | (col("fy2024_operating_ime_payment") > 0), 1).otherwise(0)
    )

print(f"Rows with impacts : {df_impacts.count():,}")
print(f"Columns           : {len(df_impacts.columns)}")

print("\nPayment impact preview:")
df_impacts.select(
    "provider_number",
    "hospital_name",
    "fy2023_operating_payment",
    "fy2024_operating_payment",
    "operating_payment_change_dollars",
    "operating_payment_change_pct",
    "wage_index_impact_category"
).show(5, truncate=False)

In [0]:
# ============================================================
# STEP 4 — ENRICH WITH HOSPITAL DIMENSION
# Join hospital info to add type, ownership, and quality ratings
# ============================================================

print("\n" + "=" * 55)
print("  JOINING HOSPITAL DIMENSION")
print("=" * 55)

df_hospital = spark.table(f"{BRONZE_DB}.hospital_info_raw")

df_enriched = df_impacts.join(
    df_hospital.select(
        col("facility_id").alias("provider_number"),
        col("hospital_type"),
        col("hospital_ownership"),
        col("emergency_services"),
        col("hospital_overall_rating"),
        col("county_parish").alias("county"),
        col("meets_criteria_for_birthing_friendly_designation").alias("birthing_friendly")
    ),
    on="provider_number",
    how="left"
)

rows_after_join = df_enriched.count()
rows_matched = df_enriched.filter(col("hospital_type").isNotNull()).count()
rows_unmatched = rows_after_join - rows_matched

print(f"Rows after join     : {rows_after_join:,}")
print(f"Matched to hospital : {rows_matched:,}")
print(f"Unmatched (no info) : {rows_unmatched:,}")

In [0]:
# ============================================================
# STEP 5 — RUN DATA QUALITY CHECKS
# Failed rows go to quarantine — never silently dropped
# ============================================================

print("\n" + "=" * 55)
print("  RUNNING DATA QUALITY CHECKS")
print("=" * 55)

rows_before_dq = df_enriched.count()

df_clean = run_dq_checks(
    df            = df_enriched,
    source_table  = f"{BRONZE_DB}.ipps_final_rule_raw",
    notebook_name = NOTEBOOK_NAME
)

clean_count = df_clean.count()
quarantine_count = rows_before_dq - clean_count

print(f"\nDQ Results:")
print(f"  Passed      : {clean_count:,}")
print(f"  Quarantined : {quarantine_count:,}")

In [0]:
# ============================================================
# STEP 6 — ADD SILVER PROCESSING METADATA
# Track when each row was processed through silver
# ============================================================

from pyspark.sql.functions import lit

df_silver = df_clean \
    .withColumn("_silver_processed_at", lit(BATCH_TIMESTAMP).cast("timestamp")) \
    .withColumn("_silver_batch_id", lit(BATCH_ID))

print(f"Silver rows ready : {df_silver.count():,}")
print(f"Total columns     : {len(df_silver.columns)}")

In [0]:
# ============================================================
# STEP 7 — WRITE TO SILVER DELTA TABLE
# ============================================================

print("\n" + "=" * 55)
print("  WRITING TO SILVER DELTA TABLE")
print("=" * 55)

target_table = f"{SILVER_DB}.ipps_payment_impacts"

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print("Running OPTIMIZE...")
spark.sql(f"OPTIMIZE {target_table}")

print(f"\nSilver table : {target_table}")
print(f"Rows written : {spark.table(target_table).count():,}")

In [0]:
# ============================================================
# STEP 8 — VERIFICATION
# ============================================================

print("\n" + "=" * 55)
print("  SILVER VERIFICATION")
print("=" * 55)

spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT provider_number) AS unique_hospitals,
        COUNT(DISTINCT _silver_batch_id) AS batches_processed,
        MIN(_silver_processed_at) AS first_processed,
        MAX(_silver_processed_at) AS last_processed,
        MAX(_fiscal_year) AS fiscal_year
    FROM {target_table}
""").show(truncate=False)

print("\nPayment impact summary:")
spark.sql(f"""
    SELECT
        wage_index_impact_category,
        COUNT(*) AS hospital_count,
        ROUND(AVG(operating_payment_change_dollars), 2) AS avg_payment_change,
        ROUND(SUM(operating_payment_change_dollars), 2) AS total_payment_change
    FROM {target_table}
    WHERE operating_payment_change_dollars IS NOT NULL
    GROUP BY wage_index_impact_category
    ORDER BY avg_payment_change DESC
""").show(truncate=False)

print("\nTop 5 hospitals with largest payment increases:")
spark.sql(f"""
    SELECT
        provider_number,
        hospital_name,
        fy2023_operating_payment,
        fy2024_operating_payment,
        operating_payment_change_dollars,
        operating_payment_change_pct
    FROM {target_table}
    WHERE operating_payment_change_dollars IS NOT NULL
    ORDER BY operating_payment_change_dollars DESC
    LIMIT 5
""").show(truncate=False)

print("\nTop 5 hospitals with largest payment decreases:")
spark.sql(f"""
    SELECT
        provider_number,
        hospital_name,
        fy2023_operating_payment,
        fy2024_operating_payment,
        operating_payment_change_dollars,
        operating_payment_change_pct
    FROM {target_table}
    WHERE operating_payment_change_dollars IS NOT NULL
    ORDER BY operating_payment_change_dollars ASC
    LIMIT 5
""").show(truncate=False)

In [0]:
# ============================================================
# STEP 9 — LOG TO AUDIT TABLE
# ============================================================

log_pipeline_run(
    notebook_name    = NOTEBOOK_NAME,
    layer            = "silver",
    status           = "SUCCESS",
    rows_read        = bronze_count,
    rows_written     = df_silver.count(),
    rows_quarantined = quarantine_count,
    message          = f"Batch {BATCH_ID} — IPPS payment impacts processed for FY2024"
)

print("\n" + "=" * 55)
print("  SILVER PROCESSING COMPLETE")
print("=" * 55)